# Academic Papers RAG System Tutorial

This notebook demonstrates how to build a complete RAG (Retrieval-Augmented Generation) system for academic research papers using LlamaIndex. We'll build it step by step with independent functions that you can run and understand individually.

## What is RAG?

RAG combines the power of:
- **Retrieval**: Finding relevant documents from a knowledge base
- **Augmented Generation**: Using retrieved context to generate informed responses

## System Components

Our RAG system will include:
1. **PDF Processing**: Extract text from academic papers
2. **Document Chunking**: Split documents into searchable segments
3. **Vector Embeddings**: Convert text to numerical representations
4. **Vector Storage**: Store embeddings in LanceDB
5. **Semantic Search**: Find relevant content for queries
6. **Query Engine**: Generate responses using retrieved context


## 🏗️ Storage Architecture: Why StorageContext Matters

This notebook uses LlamaIndex's **StorageContext** approach, which provides significant advantages over simpler vector-only storage methods. Understanding this architecture is crucial for building production-ready RAG systems.

### 📊 StorageContext vs. Simple Vector Storage

| Component | StorageContext (This Notebook) | Simple Vector Store | Benefits |
|-----------|-------------------------------|-------------------|----------|
| **Vector Store** | ✅ LanceDB embeddings | ✅ LanceDB embeddings | Fast similarity search |
| **Document Store** | ✅ Original documents preserved | ❌ Lost after processing | Full document access |
| **Index Store** | ✅ Index metadata & structure | ❌ Must rebuild index | Exact reconstruction |
| **Graph Store** | ✅ Document relationships | ❌ No relationship data | Rich context understanding |

### 🔄 Persistence & Recovery Capabilities

**With StorageContext (Our Approach):**
```python
# Save complete system state
index.storage_context.persist(persist_dir="storage/papers_index")

# Perfect restoration - identical behavior
storage_context = StorageContext.from_defaults(persist_dir="storage/papers_index")
index = load_index_from_storage(storage_context)
# 🎯 Exact same results every time!
```

**Simple Vector Store Only:**
```python
# Only vectors saved
vector_store = LanceDBVectorStore(uri="./vectors")

# Must recreate everything from scratch
index = VectorStoreIndex.from_vector_store(vector_store)
# ⚠️ May have different behavior, lost metadata
```

### 💡 Key Advantages of StorageContext

1. **🔄 Perfect Reproducibility**: Identical results across sessions - critical for research and development
2. **📦 Complete State Management**: All components preserved, not just vectors
3. **⚡ Fast Startup**: No reprocessing needed - load existing index instantly
4. **🔍 Rich Metadata**: Document relationships, source tracking, and complex queries
5. **🛠️ Development Friendly**: Iterate without rebuilding entire system
6. **🎯 Enterprise Ready**: Robust persistence for production deployments

### 📈 Storage Footprint Example

For 1000 academic papers (~500MB original PDFs):

**StorageContext Storage:**
```
storage/papers_index/
├── docstore.json          # 50MB - Original documents
├── index_store.json       # 5MB  - Index metadata  
├── graph_store.json       # 2MB  - Relationships
└── LanceDB vector files   # 200MB - Embeddings
Total: ~260MB
```

**Benefits**: Complete system restoration, full metadata, relationships preserved

**Simple Vector Storage:**
```
lancedb_data/
└── vectors.lance          # 200MB - Embeddings only
Total: ~200MB
```

**Limitations**: Must rebuild index, lost metadata, no relationships

### 🎯 When to Use StorageContext

✅ **Research & Development** - Need reproducible experiments  
✅ **Complex Documents** - Rich metadata and relationships matter  
✅ **Production Systems** - Robust persistence and recovery required  
✅ **Academic Work** - Full traceability and citation tracking  
✅ **Multi-user Systems** - Consistent experience across users  

### 🚀 Performance Impact

- **Initial Build**: ~20% slower (stores additional metadata)
- **Subsequent Loads**: 10x faster (no reprocessing needed)
- **Query Performance**: Identical to simple vector approach
- **Storage Space**: ~30% more storage for complete persistence

This tutorial demonstrates the StorageContext approach because it provides the most robust and feature-complete RAG implementation suitable for real-world applications.


## 1. Environment Setup and Configuration

First, let's set up our environment and load necessary configurations. We'll use OpenRouter for LLM access and local embeddings (no API keys needed for embeddings).


In [2]:
#!pip install -r "../requirements.txt"

In [3]:
import os
import time
from pathlib import Path
from typing import Dict, List, Optional, Tuple

from dotenv import load_dotenv

def setup_environment():
    """
    Setup environment variables and basic configuration.
    
    Returns:
        bool: Success status
    """
    # Load environment variables from .env file
    load_dotenv()
    
    # Disable tokenizer warning
    os.environ["TOKENIZERS_PARALLELISM"] = "false"
    
    # Check for required API key
    api_key = os.getenv("OPENROUTER_API_KEY")
    if not api_key:
        print("⚠️  OPENROUTER_API_KEY not found in environment variables")
        print("Please add your OpenRouter API key to a .env file")
        return False
    
    print("✓ Environment variables loaded successfully")
    return True

# Run the setup
success = setup_environment()
if success:
    print("Environment setup complete!")
else:
    print("Environment setup failed!")

✓ Environment variables loaded successfully
Environment setup complete!


## 2. Configuration Management

Let's define our system configuration directly in the notebook. This includes model settings, chunk sizes, and other parameters.


In [4]:
# Configuration parameters for the RAG system
CONFIG = {
    "llm": {
        "model": "gpt-4o",                    # OpenRouter model to use
        "temperature": 0.1                   # Temperature for response generation
    },
    "embeddings": {
        "model": "local:BAAI/bge-small-en-v1.5",  # Local embedding model (no API key needed)
        "chunk_size": 1024,                  # Size of text chunks for processing
        "chunk_overlap": 100                 # Overlap between consecutive chunks
    },
    "vector_store": {
        "type": "lancedb",                   # Vector database type
        "table_name": "academic_papers",     # Table name for storing embeddings
        "path": "storage/papers_vectordb"    # Path to vector database
    },
    "index": {
        "storage_path": "storage/papers_index",  # Path to store complete index
        "similarity_top_k": 5                    # Number of similar chunks to retrieve
    },
    "papers": {
        "folder": "../papers/agents"      # Path to academic papers folder
    }
}

def get_config(key_path: str, default_value=None):
    """
    Get configuration value using dot notation.
    
    Args:
        key_path (str): Dot-separated path to the config value (e.g., 'llm.model')
        default_value: Default value if key not found
        
    Returns:
        Configuration value or default
    """
    keys = key_path.split('.')
    value = CONFIG
    
    for key in keys:
        if isinstance(value, dict) and key in value:
            value = value[key]
        else:
            return default_value
    
    return value

# Test configuration access
llm_model = get_config("llm.model")
embedding_model = get_config("embeddings.model")
chunk_size = get_config("embeddings.chunk_size")

print(f"LLM model: {llm_model}")
print(f"Embedding model: {embedding_model}")
print(f"Chunk size: {chunk_size}")
print("✓ Configuration setup complete")


LLM model: gpt-4o
Embedding model: local:BAAI/bge-small-en-v1.5
Chunk size: 1024
✓ Configuration setup complete


## 3. LlamaIndex Settings Configuration

LlamaIndex uses global settings for embeddings, LLMs, and document processing. We'll use OpenRouter for the LLM and a local embedding model (no API key required).


In [5]:
from llama_index.core import Settings
from llama_index.llms.openrouter import OpenRouter
from llama_index.core.embeddings import resolve_embed_model
from llama_index.core.node_parser import SentenceSplitter

def configure_llamaindex_settings():
    """
    Configure LlamaIndex global settings for embeddings, LLM, and text processing.
    """
    # Set up LLM with OpenRouter
    model = get_config("llm.model")
    temperature = get_config("llm.temperature", 0.1)
    
    Settings.llm = OpenRouter(
        api_key=os.getenv("OPENROUTER_API_KEY"),
        model=model,
        temperature=temperature
    )
    print(f"✓ LLM configured: {model} (temperature: {temperature})")

    # Set up local embedding model (downloads locally first time, then cached)
    embedding_model = get_config("embeddings.model")
    Settings.embed_model = resolve_embed_model(embedding_model)
    print(f"✓ Embedding model configured: {embedding_model}")

    # Set up node parser for chunking
    chunk_size = get_config("embeddings.chunk_size")
    chunk_overlap = get_config("embeddings.chunk_overlap")
    
    Settings.node_parser = SentenceSplitter(
        chunk_size=chunk_size, 
        chunk_overlap=chunk_overlap
    )
    print(f"✓ Text chunking configured: {chunk_size} chars with {chunk_overlap} overlap")

# Configure the settings using our hardcoded config
configure_llamaindex_settings()
print("✓ LlamaIndex settings configured successfully")


/Users/vidyadharbendre/workspace/VEnV/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✓ LLM configured: gpt-4o (temperature: 0.1)
✓ Embedding model configured: local:BAAI/bge-small-en-v1.5
✓ Text chunking configured: 1024 chars with 100 overlap
✓ LlamaIndex settings configured successfully


## 4. Vector Store Setup

We'll use LanceDB as our vector database to store document embeddings. LanceDB is a fast, serverless vector database that's perfect for RAG applications.


In [6]:
from llama_index.vector_stores.lancedb import LanceDBVectorStore

def create_vector_store():
    """
    Create and configure LanceDB vector store using config settings.
    
    Returns:
        LanceDBVectorStore: Configured vector store
    """
    try:
        import lancedb
        
        # Get configuration values
        vector_db_path = get_config("vector_store.path")
        table_name = get_config("vector_store.table_name")
        
        # Create storage directory
        Path(vector_db_path).parent.mkdir(parents=True, exist_ok=True)
        
        # Connect to LanceDB
        db = lancedb.connect(str(vector_db_path))
        print(f"✓ Connected to LanceDB at: {vector_db_path}")
        
        # Create vector store
        vector_store = LanceDBVectorStore(
            uri=str(vector_db_path), 
            table_name=table_name
        )
        print(f"✓ LanceDB vector store created (table: {table_name})")
        
        return vector_store
        
    except Exception as e:
        print(f"Error creating vector store: {e}")
        return None

# Create the vector store using config
vector_store = create_vector_store()
if vector_store:
    print("✓ Vector store setup complete")
else:
    print("❌ Vector store setup failed")


✓ Connected to LanceDB at: storage/papers_vectordb
✓ LanceDB vector store created (table: academic_papers)
✓ Vector store setup complete


## 5. PDF Processing and Document Loading

Now we'll create functions to load and process PDF files. We'll use LlamaIndex's native `SimpleDirectoryReader` which can handle PDFs directly without needing a custom processor.


In [7]:
from llama_index.core import SimpleDirectoryReader

def load_papers_from_folder() -> List:
    """
    Load and process all PDF papers from the configured folder using LlamaIndex's native loader.
    
    Returns:
        List[Document]: Processed documents ready for indexing
    """
    papers_folder = get_config("papers.folder")
    print(f"Loading papers from: {papers_folder}")
    
    papers_path = Path(papers_folder)
    if not papers_path.exists():
        print(f"Papers folder does not exist: {papers_path}")
        return []
    
    # Use LlamaIndex's SimpleDirectoryReader to load PDFs
    # This natively handles PDF parsing, text extraction, and metadata
    documents = SimpleDirectoryReader(papers_folder).load_data()
    
    print(f"✓ Loaded {len(documents)} documents")
    return documents

# Load the papers using config
documents = load_papers_from_folder()
if documents:
    print(f"Successfully loaded {len(documents)} documents")
    print(f"First document preview: {documents[0].text[:200]}...")
    print(f"First document metadata: {documents[0].metadata}")
else:
    print("No documents loaded")


Loading papers from: ../papers/agents
✓ Loaded 229 documents
Successfully loaded 229 documents
First document preview: AI Agents vs. Agentic AI: A Conceptual
Taxonomy, Applications and Challenges
Ranjan Sapkota∗‡, Konstantinos I. Roumeliotis †, Manoj Karkee ∗‡
∗Cornell University, Department of Environmental and Biolo...
First document metadata: {'page_label': '1', 'file_name': 'AI_Agents_vs_Agentic_AI.pdf', 'file_path': '/Users/vidyadharbendre/workspace/VEnV/ai-accelerator/Day_6/session_2/llamaindex_rag/../papers/agents/AI_Agents_vs_Agentic_AI.pdf', 'file_type': 'application/pdf', 'file_size': 3196781, 'creation_date': '2025-09-21', 'last_modified_date': '2025-09-21'}


## 6. Creating the Vector Index

The vector index is the core of our RAG system. It chunks documents, generates embeddings, and stores them in the vector database for efficient similarity search.


In [22]:
import hashlib
import json
import logging
import shutil
import time
from pathlib import Path
from typing import List, Optional

from llama_index.core import StorageContext, VectorStoreIndex, load_index_from_storage

LOG = logging.getLogger("rag.index")
if not LOG.handlers:
    logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")


def _config_fingerprint() -> str:
    """Hash only the settings that affect the built index."""
    relevant = {
        "embeddings.model": get_config("embeddings.model"),
        "embeddings.chunk_size": get_config("embeddings.chunk_size"),
        "embeddings.chunk_overlap": get_config("embeddings.chunk_overlap"),
        "vector_store.type": get_config("vector_store.type"),
        "vector_store.table_name": get_config("vector_store.table_name"),
        "vector_store.path": str(get_config("vector_store.path")),
        "index.similarity_top_k": get_config("index.similarity_top_k"),
    }
    return hashlib.sha256(json.dumps(relevant, sort_keys=True).encode("utf-8")).hexdigest()


def _reset_persisted_index(index_path: Path) -> None:
    """Delete LlamaIndex persisted files."""
    shutil.rmtree(index_path, ignore_errors=True)
    index_path.mkdir(parents=True, exist_ok=True)


def _drop_lancedb_table_safely() -> None:
    """Drop the LanceDB table if it exists. Safe to call even if missing."""
    try:
        import lancedb  # type: ignore
        db_path = get_config("vector_store.path")
        table_name = get_config("vector_store.table_name")
        db = lancedb.connect(db_path)
        if table_name in db.table_names():
            db.drop_table(table_name)
            LOG.info("Dropped LanceDB table: %s", table_name)
        else:
            LOG.info("LanceDB table %s not found; nothing to drop.", table_name)
    except Exception as e:
        LOG.warning("Could not drop LanceDB table cleanly: %s", e)


def _write_meta(meta_file: Path, fp: str, status: str = "ok") -> None:
    try:
        meta_file.write_text(json.dumps({"fingerprint": fp, "status": status, "ts": time.time()}, indent=2))
    except Exception as e:
        LOG.warning("Failed writing meta.json: %s", e)


def _read_meta_fingerprint(meta_file: Path) -> Optional[str]:
    try:
        if meta_file.exists():
            data = json.loads(meta_file.read_text())
            return data.get("fingerprint")
    except Exception as e:
        LOG.warning("Failed reading meta.json: %s", e)
    return None


def create_vector_index(
    documents: List,
    vector_store,
    force_rebuild: bool = False
) -> Optional[VectorStoreIndex]:
    """
    Create or load a vector index from documents using config settings.

    Args:
        documents (List): Documents to index
        vector_store: LanceDB vector store (already constructed)
        force_rebuild (bool): Force rebuild even if index exists

    Returns:
        VectorStoreIndex | None
    """
    index_storage_path = get_config("index.storage_path")
    index_path = Path(index_storage_path)
    index_path.mkdir(parents=True, exist_ok=True)

    index_store_file = index_path / "index_store.json"
    meta_file = index_path / "meta.json"
    desired_fp = _config_fingerprint()

    # Decide whether to load existing
    should_load = index_store_file.exists() and not force_rebuild

    if should_load:
        current_fp = _read_meta_fingerprint(meta_file)
        if current_fp is None:
            LOG.info("No meta fingerprint found; rebuilding to be safe.")
            should_load = False
        elif current_fp != desired_fp:
            LOG.info("Config changed since last build; rebuilding index.")
            should_load = False

    if force_rebuild:
        LOG.info("Force rebuild requested; removing persisted index and dropping LanceDB table.")
        _reset_persisted_index(index_path)
        _drop_lancedb_table_safely()
        should_load = False  # ensure rebuild path

    if should_load:
        LOG.info("📁 Loading existing index...")
        try:
            storage_context = StorageContext.from_defaults(
                persist_dir=str(index_path),
                vector_store=vector_store
            )
            index = load_index_from_storage(storage_context)
            LOG.info("✓ Successfully loaded existing index")
            return index
        except Exception as e:
            LOG.warning("⚠️  Error loading existing index: %s; rebuilding...", e)

    # Rebuild path
    if not documents:
        LOG.error("❌ No documents to index")
        return None

    LOG.info("🔨 Creating new vector index...")
    t0 = time.time()

    storage_context = StorageContext.from_defaults(vector_store=vector_store)
    index = VectorStoreIndex.from_documents(
        documents,
        storage_context=storage_context,
        show_progress=True
    )
    LOG.info("✓ Index created in %.2f seconds", time.time() - t0)

    LOG.info("💾 Saving index to storage...")
    index.storage_context.persist(persist_dir=str(index_path))
    _write_meta(meta_file, fp=desired_fp, status="ok")
    LOG.info("✓ Index saved successfully")

    return index


# -------- Usage (unchanged except the flag) --------
index = create_vector_index(
    documents=documents,
    vector_store=vector_store,
    force_rebuild=False  # set True to force a clean rebuild
)

if index:
    print("✓ Vector index ready for querying")
else:
    print("❌ Failed to create vector index")


2025-09-23 12:24:37,124 - INFO - No meta fingerprint found; rebuilding to be safe.
2025-09-23 12:24:37,125 - INFO - 🔨 Creating new vector index...
Generating embeddings: 100%|██████████| 412/412 [00:04<00:00, 96.59it/s] 
2025-09-23 12:24:42,183 - INFO - ✓ Index created in 5.06 seconds
2025-09-23 12:24:42,184 - INFO - 💾 Saving index to storage...
2025-09-23 12:24:42,186 - INFO - ✓ Index saved successfully


✓ Vector index ready for querying


## 7. Setting Up the Query Engine

The query engine combines a retriever (to find relevant documents) with an LLM (to generate responses). This is where the "Augmented Generation" part of RAG happens.


In [23]:
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.retrievers import VectorIndexRetriever

def setup_query_engine(index):
    """
    Setup the query engine for semantic search and response generation using config settings.
    
    Args:
        index: The vector index to query
        
    Returns:
        RetrieverQueryEngine: Configured query engine
    """
    if not index:
        print("❌ Index not available. Please create index first.")
        return None
    
    try:
        # Get similarity top k from config
        similarity_top_k = get_config("index.similarity_top_k")
        
        # Create retriever - this finds the most similar document chunks
        retriever = VectorIndexRetriever(
            index=index,
            similarity_top_k=similarity_top_k,
        )
        print(f"✓ Retriever configured to find top {similarity_top_k} similar chunks")
        
        # Create query engine - this combines retrieval with LLM generation
        query_engine = RetrieverQueryEngine(retriever=retriever)
        print("✓ Query engine setup successfully")
        
        return query_engine
        
    except Exception as e:
        print(f"❌ Error setting up query engine: {e}")
        return None

# Setup the query engine using config
query_engine = setup_query_engine(index)

if query_engine:
    print("🚀 RAG system is ready for queries!")
else:
    print("❌ Failed to setup query engine")


✓ Retriever configured to find top 5 similar chunks
✓ Query engine setup successfully
🚀 RAG system is ready for queries!


In [24]:
def extract_paper_title_from_text(text: str, max_length: int = 200) -> str:
    """
    Extract the paper title from the document text.
    
    Args:
        text (str): Document text content
        max_length (int): Maximum length for title extraction
        
    Returns:
        str: Extracted title or fallback
    """
    if not text:
        return "Unknown Title"
    
    # Split into lines and clean them
    lines = [line.strip() for line in text.split('\n') if line.strip()]
    
    if not lines:
        return "Unknown Title"
    
    # Look for title patterns - usually the first substantial line
    # Skip very short lines, page numbers, headers
    for line in lines[:10]:  # Check first 10 lines
        # Skip lines that look like headers, page numbers, or metadata
        if (len(line) > 15 and 
            not line.isdigit() and 
            not line.startswith(('Page', 'arXiv:', 'doi:', 'http', 'www')) and
            not all(c.isupper() or c.isspace() for c in line) and  # Skip all-caps headers
            '.' in line or len(line) > 30):  # Likely a title if it has punctuation or is long
            
            # Clean up the title
            title = line.strip()
            
            # Remove common prefixes/suffixes
            prefixes_to_remove = ['Title:', 'Abstract:', 'Paper:', 'Research:']
            for prefix in prefixes_to_remove:
                if title.startswith(prefix):
                    title = title[len(prefix):].strip()
            
            # Truncate if too long
            if len(title) > max_length:
                title = title[:max_length].strip() + "..."
            
            return title
    
    # Fallback: use first non-empty line, truncated
    first_line = lines[0] if lines else "Unknown Title"
    if len(first_line) > max_length:
        first_line = first_line[:max_length].strip() + "..."
    
    return first_line

def extract_paper_authors_from_text(text: str) -> str:
    """
    Extract authors from the document text.
    
    Args:
        text (str): Document text content
        
    Returns:
        str: Extracted authors or "Unknown Authors"
    """
    if not text:
        return "Unknown Authors"
    
    lines = [line.strip() for line in text.split('\n') if line.strip()]
    
    # Look for author patterns in first 20 lines
    for i, line in enumerate(lines[:20]):
        # Skip the title line (usually first substantial line)
        if i == 0:
            continue
            
        # Look for author patterns
        if (len(line) > 5 and 
            not line.isdigit() and
            not line.startswith(('Abstract', 'Introduction', 'Page', 'arXiv:', 'doi:', 'http')) and
            ('University' in line or 'Institute' in line or 
             ',' in line or 'Department' in line or
             '@' in line or  # Email addresses often indicate authors
             any(char.isupper() for char in line))):  # Names often have capitals
            
            # Clean up author line
            authors = line.strip()
            
            # Remove common prefixes
            prefixes_to_remove = ['Authors:', 'By:', 'Author:']
            for prefix in prefixes_to_remove:
                if authors.startswith(prefix):
                    authors = authors[len(prefix):].strip()
            
            # Truncate if too long
            if len(authors) > 150:
                authors = authors[:150].strip() + "..."
            
            return authors
    
    return "Unknown Authors"

print("📝 Title and Author extraction functions loaded successfully!")


📝 Title and Author extraction functions loaded successfully!


## 8. Search and Query Functions

Now let's create functions to search through our academic papers and extract detailed information about sources and metadata.


In [25]:
# Updated list_indexed_papers function with title extraction
def list_indexed_papers_improved(documents: List) -> List[Dict[str, any]]:
    """
    List all papers that have been indexed with their metadata.
    Extracts actual paper titles and authors from document content.
    
    Args:
        documents (List): List of loaded documents
        
    Returns:
        List[Dict[str, any]]: List of paper information
    """
    papers = []
    processed_files = set()  # Track unique files to avoid duplicates
    
    for doc in documents:
        try:
            metadata = doc.metadata
            file_path = metadata.get("file_path", "")
            file_name = Path(file_path).stem if file_path else "Unknown"
            
            # Skip if we've already processed this file
            if file_path in processed_files:
                continue
            processed_files.add(file_path)
            
            # Extract title and authors from document text
            doc_text = doc.text if hasattr(doc, 'text') else ""
            extracted_title = extract_paper_title_from_text(doc_text)
            extracted_authors = extract_paper_authors_from_text(doc_text)
            
            paper_info = {
                "file_name": file_name,
                "file_path": file_path,
                "title": extracted_title,
                "authors": extracted_authors,
                "page_count": metadata.get("page_count", 0),
                "file_size": metadata.get("file_size", 0),
                "file_size_mb": round(metadata.get("file_size", 0) / (1024 * 1024), 2) if metadata.get("file_size") else 0,
                "total_pages": metadata.get("total_pages", "Unknown"),
                "page_label": metadata.get("page_label", ""),
            }
            
            papers.append(paper_info)
            
        except Exception as e:
            print(f"Error processing document: {e}")
    
    return papers

# Re-list papers with improved title extraction
print("🔄 Re-processing papers with improved title extraction...")
papers_list_improved = list_indexed_papers_improved(documents)

print(f"📚 Found {len(papers_list_improved)} unique papers in the index:")
print("=" * 80)

for i, paper in enumerate(papers_list_improved[:10], 1):  # Show first 10 papers
    print(f"{i}. 📄 {paper['title']}")
    print(f"   👥 Authors: {paper['authors']}")
    print(f"   📁 File: {paper['file_name']}")
    print(f"   💾 Size: {paper['file_size_mb']} MB")
    if paper.get('page_label'):
        print(f"   📖 Page: {paper['page_label']}")
    print("-" * 60)

print(f"\n✅ Successfully extracted titles for {len(papers_list_improved)} papers!")
if len(papers_list_improved) > 10:
    print(f"📝 Showing first 10 papers. Total: {len(papers_list_improved)} papers available.")


🔄 Re-processing papers with improved title extraction...
📚 Found 5 unique papers in the index:
1. 📄 AI Agents vs. Agentic AI: A Conceptual
   👥 Authors: Taxonomy, Applications and Challenges
   📁 File: AI_Agents_vs_Agentic_AI
   💾 Size: 3.05 MB
   📖 Page: 1
------------------------------------------------------------
2. 📄 THE LANDSCAPE OF EMERGING AI AGENT ARCHITECTURES
   👥 Authors: FOR REASONING , PLANNING , AND TOOL CALLING : A S URVEY
   📁 File: Emerging_Agent_Architectures
   💾 Size: 1.58 MB
   📖 Page: 1
------------------------------------------------------------
3. 📄 From LLM Reasoning to Autonomous AI Agents:
   👥 Authors: From LLM Reasoning to Autonomous AI Agents:
   📁 File: LLMReasoning_to_Autonomous_Agents
   💾 Size: 16.21 MB
   📖 Page: 1
------------------------------------------------------------
4. 📄 The Rise and Potential of Large Language Model
   👥 Authors: Based Agents: A Survey
   📁 File: Rise_and_Potential_LLM_Agents
   💾 Size: 6.52 MB
   📖 Page: 1
----------------

In [26]:
# Enhanced search function with improved metadata extraction
def search_papers_improved(query_engine, query: str, include_metadata: bool = True) -> Dict[str, any]:
    """
    Search for relevant papers based on the query with improved title extraction.
    
    Args:
        query_engine: The configured query engine
        query (str): Search query
        include_metadata (bool): Whether to include detailed metadata
        
    Returns:
        Dict[str, any]: Search results with response and sources
    """
    if not query_engine:
        return {
            "success": False,
            "error": "Query engine not initialized.",
            "response": "",
            "sources": [],
        }
    
    try:
        print(f"🔍 Searching for: '{query}'")
        start_time = time.time()
        
        # Query the RAG system
        response = query_engine.query(query)
        
        end_time = time.time()
        
        # Extract source information from retrieved nodes with title extraction
        sources = []
        if hasattr(response, "source_nodes"):
            for node in response.source_nodes:
                # Extract title from the node text
                node_text = node.text if hasattr(node, 'text') else ""
                extracted_title = extract_paper_title_from_text(node_text, max_length=100)
                extracted_authors = extract_paper_authors_from_text(node_text)
                
                source_info = {
                    "text": (
                        node.text[:500] + "..."
                        if len(node.text) > 500
                        else node.text
                    ),
                    "score": getattr(node, "score", 0.0),
                    "extracted_title": extracted_title,
                    "extracted_authors": extracted_authors,
                }
                
                # Add metadata if available and requested
                if include_metadata and hasattr(node, "metadata"):
                    metadata = node.metadata
                    source_info.update({
                        "file_name": metadata.get("file_name", "Unknown"),
                        "file_path": metadata.get("file_path", ""),
                        "page_label": metadata.get("page_label", ""),
                        "file_size_mb": round(metadata.get("file_size", 0) / (1024 * 1024), 2) if metadata.get("file_size") else 0,
                    })
                
                sources.append(source_info)
        
        result = {
            "success": True,
            "response": str(response),
            "sources": sources,
            "query": query,
            "search_time": end_time - start_time,
            "num_sources": len(sources),
        }
        
        print(f"✓ Search completed in {end_time - start_time:.2f} seconds")
        print(f"📚 Found {len(sources)} relevant sources")
        
        return result
        
    except Exception as e:
        print(f"❌ Error during search: {e}")
        return {"success": False, "error": str(e), "response": "", "sources": []}

print("🔍 Enhanced search function with title extraction loaded!")


🔍 Enhanced search function with title extraction loaded!


In [27]:
# Test the improved search with title extraction
def ask_question_improved(query_engine, question: str, show_sources: bool = True):
    """
    Ask a custom question to the RAG system with improved title extraction.
    
    Args:
        query_engine: The configured query engine
        question (str): Your question about the papers
        show_sources (bool): Whether to display source information
    """
    print(f"❓ Question: {question}")
    print("=" * 80)
    
    result = search_papers_improved(query_engine, question, include_metadata=True)
    
    if result["success"]:
        print(f"💡 Answer:")
        print(result["response"])
        print(f"\n📊 Search completed in {result['search_time']:.2f} seconds")
        print(f"📚 Found {result['num_sources']} relevant sources")
        
        if show_sources and result["sources"]:
            print(f"\n📖 Source Details:")
            print("-" * 60)
            for i, source in enumerate(result["sources"], 1):
                print(f"\n{i}. 📄 Paper: {source.get('extracted_title', 'Unknown Title')}")
                print(f"   👥 Authors: {source.get('extracted_authors', 'Unknown Authors')}")
                print(f"   📁 File: {source.get('file_name', 'Unknown')}")
                if source.get('page_label'):
                    print(f"   📖 Page: {source['page_label']}")
                print(f"   🎯 Relevance Score: {source.get('score', 0):.3f}")
                print(f"   📝 Text Preview: {source['text'][:200]}...")
                
    else:
        print(f"❌ Error: {result['error']}")

# Test with the improved version
test_question = "What are the main architectural patterns for agent systems?"

if query_engine:
    print("🧪 Testing improved search with title extraction:")
    print("=" * 80)
    ask_question_improved(query_engine, test_question, show_sources=True)
else:
    print("❌ Query engine not available")


🧪 Testing improved search with title extraction:
❓ Question: What are the main architectural patterns for agent systems?
🔍 Searching for: 'What are the main architectural patterns for agent systems?'


2025-09-23 12:34:11,620 - INFO - HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"


✓ Search completed in 9.55 seconds
📚 Found 3 relevant sources
💡 Answer:
The main architectural patterns for agent systems include foundational components such as perception, reasoning, action, and learning. These systems often evolve from simple, modular designs to more complex orchestration frameworks. Enhancements in these architectures can involve persistent memory, meta-agent coordination, multi-agent planning loops, and semantic communication protocols. These patterns support the transition from isolated AI Agents to collaborative, distributed ecosystems of interacting agents, enabling capabilities like goal decomposition, subtask assignment, and dynamic adaptation to changing contexts.

📊 Search completed in 9.55 seconds
📚 Found 3 relevant sources

📖 Source Details:
------------------------------------------------------------

1. 📄 Paper: AI enhances this base by integrating advanced components
   👥 Authors: AI enhances this base by integrating advanced components
   📁 File: AI_A

In [28]:
def search_papers(query_engine, query: str, include_metadata: bool = True) -> Dict[str, any]:
    """
    Search for relevant papers based on the query.
    
    Args:
        query_engine: The configured query engine
        query (str): Search query
        include_metadata (bool): Whether to include detailed metadata
        
    Returns:
        Dict[str, any]: Search results with response and sources
    """
    if not query_engine:
        return {
            "success": False,
            "error": "Query engine not initialized.",
            "response": "",
            "sources": [],
        }
    
    try:
        print(f"🔍 Searching for: '{query}'")
        start_time = time.time()
        
        # Query the RAG system
        response = query_engine.query(query)
        
        end_time = time.time()
        
        # Extract source information from retrieved nodes
        sources = []
        if hasattr(response, "source_nodes"):
            for node in response.source_nodes:
                source_info = {
                    "text": (
                        node.text[:500] + "..."
                        if len(node.text) > 500
                        else node.text
                    ),
                    "score": getattr(node, "score", 0.0),
                }
                
                # Add metadata if available and requested
                if include_metadata and hasattr(node, "metadata"):
                    metadata = node.metadata
                    source_info.update({
                        "file_name": metadata.get("file_name", "Unknown"),
                        "title": metadata.get("title", "Unknown Title"),
                        "authors": metadata.get("authors", "Unknown Authors"),
                        "page_count": metadata.get("page_count", 0),
                        "has_abstract": metadata.get("has_abstract", False),
                    })
                
                sources.append(source_info)
        
        result = {
            "success": True,
            "response": str(response),
            "sources": sources,
            "query": query,
            "search_time": end_time - start_time,
            "num_sources": len(sources),
        }
        
        print(f"✓ Search completed in {end_time - start_time:.2f} seconds")
        print(f"📚 Found {len(sources)} relevant sources")
        
        return result
        
    except Exception as e:
        print(f"❌ Error during search: {e}")
        return {"success": False, "error": str(e), "response": "", "sources": []}

# Test the search function with a sample query
test_query = "What are the main types of AI agents discussed in these papers?"
result = search_papers(query_engine, test_query)

if result["success"]:
    print(f"\n📝 Response Preview: {result['response']}")
    print(f"📊 Number of sources: {result['num_sources']}")
    print(f"⏱️  Search time: {result['search_time']:.2f} seconds")
else:
    print(f"❌ Search failed: {result['error']}")


🔍 Searching for: 'What are the main types of AI agents discussed in these papers?'


2025-09-23 12:34:58,958 - INFO - HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"


✓ Search completed in 3.52 seconds
📚 Found 3 relevant sources

📝 Response Preview: The main types of AI agents discussed are AI Agents and Agentic AI. AI Agents are applied in areas such as knowledge retrieval, email automation, and report summarization. Agentic AI is used in domains like research assistants, robotic swarms, and strategic business planning.
📊 Number of sources: 3
⏱️  Search time: 3.52 seconds


## 9. Paper Information and Metadata

Let's create functions to list and get detailed information about the papers in our index.


In [33]:
from __future__ import annotations

import os
import re
from pathlib import Path
from typing import Dict, List, Optional, Tuple

# If using LlamaIndex v0.10+, Document has .metadata (dict-like)
# and chunked Nodes may carry the same file_path repeated.

try:
    from pypdf import PdfReader  # lightweight and reliable for page count
except Exception:
    PdfReader = None  # handled gracefully

# --------- PDF helpers ---------

def _pdf_page_count(path: Path) -> Optional[int]:
    """Return page count for a PDF, or None if unreadable/not a PDF."""
    try:
        if not path.exists() or not path.is_file():
            return None
        if path.suffix.lower() != ".pdf":
            return None
        if PdfReader is None:
            return None
        reader = PdfReader(str(path))
        if getattr(reader, "is_encrypted", False):
            try:
                reader.decrypt("")  # type: ignore[arg-type]
            except Exception:
                return None
        return len(reader.pages)
    except Exception:
        return None

# --------- File size helpers (robust) ---------

SIZE_RE = re.compile(r"^\s*([0-9]*\.?[0-9]+)\s*(bytes?|kb|mb|gb)\s*$", re.I)

def _parse_size_to_bytes(val) -> Optional[int]:
    """Parse '3.05 MB', '312 KB', '123456 bytes', or numeric-like strings to bytes."""
    if val is None:
        return None
    if isinstance(val, (int, float)):
        return int(val)
    if not isinstance(val, str):
        return None
    s = val.strip()
    # Try explicit unit pattern first
    m = SIZE_RE.match(s)
    if m:
        num = float(m.group(1))
        unit = m.group(2).lower()
        if unit.startswith("byte"):
            return int(num)
        if unit == "kb":
            return int(num * 1024)
        if unit == "mb":
            return int(num * 1024 * 1024)
        if unit == "gb":
            return int(num * 1024 * 1024 * 1024)
        return None
    # Fallback: raw integer string
    try:
        return int(s)
    except Exception:
        return None

def _try_stat_multiple_locations(file_path_str: str) -> Optional[int]:
    """
    Try stat() via several resolution strategies and return st_size if found.
    """
    candidates: List[Path] = []
    p0 = Path(file_path_str)
    candidates += [p0, p0.expanduser(), p0.expanduser().resolve(strict=False)]

    # Try relative to configured papers folder if available
    try:
        papers_root = Path(get_config("papers.folder")).expanduser()
        candidates += [
            (papers_root / p0).resolve(strict=False),
            (papers_root / p0.name).resolve(strict=False),
        ]
    except Exception:
        pass

    seen = set()
    for cand in candidates:
        try:
            if not isinstance(cand, Path):
                continue
            key = str(cand)
            if key in seen:
                continue
            seen.add(key)
            if cand.exists() and cand.is_file():
                return cand.stat().st_size
        except Exception:
            continue
    return None

def _file_size_bytes_from_metadata(md: dict) -> Optional[int]:
    """
    Best-effort size detection:
    1) stat from disk using file_path/source/filepath
    2) parse md['file_size'] if it's a string/numeric
    3) parse other common keys like 'size'
    """
    file_path = md.get("file_path") or md.get("source") or md.get("filepath")
    if file_path:
        sz = _try_stat_multiple_locations(file_path)
        if sz is not None:
            return sz

    if "file_size" in md and md["file_size"] not in (None, "", 0):
        parsed = _parse_size_to_bytes(md["file_size"])
        if parsed is not None:
            return parsed

    if "size" in md and md["size"] not in (None, "", 0):
        parsed = _parse_size_to_bytes(md["size"])
        if parsed is not None:
            return parsed

    return None

# --------- Enrichment & Listing ---------

def enrich_documents_metadata(documents: List) -> None:
    """
    Mutates each document's metadata to add/normalize:
      - file_path (normalized), file_name
      - total_pages (PDF)
      - file_size (bytes), file_size_mb (float, 2 decimals)
      - title fallback
    """
    for doc in documents:
        try:
            md = getattr(doc, "metadata", {}) or {}

            # Normalize canonical path keys
            raw_path = md.get("file_path") or md.get("source") or md.get("filepath") or ""
            if raw_path:
                p = Path(raw_path).expanduser().resolve(strict=False)
                md["file_path"] = str(p)
                md.setdefault("file_name", p.stem)
            else:
                md.setdefault("file_name", md.get("title", "Unknown"))

            # --- FILE SIZE (robust) ---
            sz_bytes = _file_size_bytes_from_metadata(md)
            # Overwrite if unknown or non-positive or non-int
            if sz_bytes is not None and (not isinstance(md.get("file_size"), int) or md.get("file_size") <= 0):
                md["file_size"] = int(sz_bytes)
            # Always compute/update MB if bytes known
            if isinstance(md.get("file_size"), int) and md["file_size"] > 0:
                md["file_size_mb"] = round(md["file_size"] / (1024 * 1024), 2)
            else:
                md["file_size_mb"] = 0  # normalize to 0 if unknown

            # --- TOTAL PAGES (PDF only) ---
            if md.get("total_pages") in (None, 0, "Unknown"):
                fpath = md.get("file_path")
                pages = _pdf_page_count(Path(fpath)) if fpath else None
                if pages is not None:
                    md["total_pages"] = pages

            # Title fallback
            md.setdefault("title", md.get("file_name", "Unknown"))

            # Write back
            doc.metadata = md
        except Exception as e:
            print(f"Warning: could not enrich metadata for a document: {e}")

def list_indexed_papers_grouped(documents: List) -> List[Dict[str, object]]:
    """
    Groups documents by file_path so each paper appears once.
    Prefers enriched metadata if available.
    """
    grouped: Dict[str, Dict[str, object]] = {}

    for doc in documents:
        try:
            md = getattr(doc, "metadata", {}) or {}
            file_path = md.get("file_path") or md.get("source") or ""
            file_name = md.get("file_name") or (Path(file_path).stem if file_path else "Unknown")

            key = file_path or file_name  # fall back to name if path missing
            item = grouped.get(key, {
                "file_name": file_name,
                "file_path": file_path,
                "title": md.get("title", file_name),
                "authors": md.get("authors", "Unknown"),
                "total_pages": md.get("total_pages", md.get("page_count", "Unknown")),
                "file_size_mb": md.get("file_size_mb", 0),
            })

            # Prefer non-unknown values as we encounter better metadata
            def _prefer(old, new, unknown_tokens={"Unknown", None, 0}):
                return new if new not in unknown_tokens else old

            item["title"] = _prefer(item.get("title"), md.get("title"))
            item["authors"] = _prefer(item.get("authors"), md.get("authors"))
            item["total_pages"] = _prefer(item.get("total_pages"), md.get("total_pages"))
            item["file_size_mb"] = _prefer(item.get("file_size_mb"), md.get("file_size_mb"))

            grouped[key] = item
        except Exception as e:
            print(f"Warning: could not aggregate a document: {e}")

    # Return sorted list by filename
    return sorted(grouped.values(), key=lambda x: str(x.get("file_name", "")))

# ----- Use it like this -----

# 1) Enrich in-place (adds total_pages, size, etc.)
enrich_documents_metadata(documents)

# 2) Build grouped list (no duplicates)
papers_list = list_indexed_papers_grouped(documents)

print(f"📚 Found {len(papers_list)} unique papers in the index:")
print("=" * 60)
for i, paper in enumerate(papers_list, 1):
    print(f"{i}. {paper['file_name']}")
    print(f"   File Path: {paper.get('file_path', '')}")
    print(f"   Total Pages: {paper.get('total_pages', 'Unknown')}")
    print(f"   Size: {paper.get('file_size_mb', 0)} MB")
    print("-" * 40)


📚 Found 5 unique papers in the index:
1. AI_Agents_vs_Agentic_AI.pdf
   File Path: /Users/vidyadharbendre/workspace/VEnV/ai-accelerator/Day_6/session_2/papers/agents/AI_Agents_vs_Agentic_AI.pdf
   Total Pages: 32
   Size: 3.05 MB
----------------------------------------
2. Emerging_Agent_Architectures.pdf
   File Path: /Users/vidyadharbendre/workspace/VEnV/ai-accelerator/Day_6/session_2/papers/agents/Emerging_Agent_Architectures.pdf
   Total Pages: 13
   Size: 1.58 MB
----------------------------------------
3. LLMReasoning_to_Autonomous_Agents.pdf
   File Path: /Users/vidyadharbendre/workspace/VEnV/ai-accelerator/Day_6/session_2/papers/agents/LLMReasoning_to_Autonomous_Agents.pdf
   Total Pages: 44
   Size: 16.21 MB
----------------------------------------
4. Rise_and_Potential_LLM_Agents.pdf
   File Path: /Users/vidyadharbendre/workspace/VEnV/ai-accelerator/Day_6/session_2/papers/agents/Rise_and_Potential_LLM_Agents.pdf
   Total Pages: 86
   Size: 6.52 MB
----------------------------

## 11. Advanced Query Examples

Now let's test our RAG system with various types of research queries to demonstrate its capabilities.


In [34]:
def run_example_queries(query_engine):
    """
    Run a series of example queries to demonstrate RAG capabilities.
    
    Args:
        query_engine: The configured query engine
    """
    example_queries = [
        {
            "category": "Agent Types",
            "query": "What are the main types of AI agents discussed in these papers?",
        },
        {
            "category": "Technical Comparison", 
            "query": "How do LLM-based agents differ from traditional AI agents?",
        },
        {
            "category": "Challenges",
            "query": "What are the current challenges in developing autonomous agents?",
        },
        {
            "category": "Evaluation",
            "query": "What evaluation methods are used for AI agents?",
        },
        {
            "category": "Architecture",
            "query": "Describe the common architectural patterns for agent systems.",
        },
        {
            "category": "Applications",
            "query": "What are the practical applications of AI agents mentioned in the literature?",
        },
    ]
    
    print("🧪 Running Example Queries")
    print("=" * 60)
    
    for i, example in enumerate(example_queries, 1):
        print(f"\n{i}. {example['category']}")
        print(f"Q: {example['query']}")
        print("-" * 50)
        
        result = search_papers(query_engine, example["query"])
        
        if result["success"]:
            # Display truncated response
            response = result["response"]
            if len(response) > 400:
                response = response[:400] + "..."
                
            print(f"A: {response}")
            print(f"📚 Sources: {result['num_sources']} | ⏱️  Time: {result['search_time']:.2f}s")
        else:
            print(f"❌ Error: {result['error']}")
        
        print()

# Run the example queries
if query_engine:
    run_example_queries(query_engine)
else:
    print("❌ Query engine not available for examples")


🧪 Running Example Queries

1. Agent Types
Q: What are the main types of AI agents discussed in these papers?
--------------------------------------------------
🔍 Searching for: 'What are the main types of AI agents discussed in these papers?'


2025-09-23 12:45:33,820 - INFO - HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"


✓ Search completed in 2.40 seconds
📚 Found 3 relevant sources
A: The main types of AI agents discussed are AI Agents and Agentic AI. AI Agents are applied in areas such as knowledge retrieval, email automation, and report summarization. Agentic AI is used in domains like research assistants, robotic swarms, and strategic business planning.
📚 Sources: 3 | ⏱️  Time: 2.40s


2. Technical Comparison
Q: How do LLM-based agents differ from traditional AI agents?
--------------------------------------------------
🔍 Searching for: 'How do LLM-based agents differ from traditional AI agents?'


2025-09-23 12:45:35,509 - INFO - HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"


✓ Search completed in 4.56 seconds
📚 Found 3 relevant sources
A: LLM-based agents differ from traditional AI agents primarily in their foundation and capabilities. Traditional AI agents often focus on specific algorithms or training strategies to enhance performance on particular tasks. In contrast, LLM-based agents leverage large language models as their foundation, which provides them with versatile capabilities that can be adapted to diverse scenarios. This ...
📚 Sources: 3 | ⏱️  Time: 4.56s


3. Challenges
Q: What are the current challenges in developing autonomous agents?
--------------------------------------------------
🔍 Searching for: 'What are the current challenges in developing autonomous agents?'


...

✓ Search completed in 4.69 seconds
📚 Found 3 relevant sources
A: AI agents are evaluated using a variety of methods that include benchmark-based evaluations, safety and alignment audits, and LLM-as-a-judge approaches. Benchmark-based evaluations involve structured tasks with standardized metrics, focusing on aspects like task completion, reasoning quality, and generalization ability. Specific benchmarks assess tool usage, web navigation, and multi-agent collabo...
📚 Sources: 3 | ⏱️  Time: 4.69s


5. Architecture
Q: Describe the common architectural patterns for agent systems.
--------------------------------------------------
🔍 Searching for: 'Describe the common architectural patterns for agent systems.'


2025-09-23 12:45:50,566 - INFO - HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"


✓ Search completed in 3.34 seconds
📚 Found 3 relevant sources
A: Agent systems commonly follow two primary architectural patterns: vertical and horizontal architectures. 

In vertical architectures, there is a lead agent that other agents report to, creating a clear hierarchy and division of labor. The lead agent may be the sole communicator with the other agents, or there may be a shared conversation among all agents.

Horizontal architectures treat all agents...
📚 Sources: 3 | ⏱️  Time: 3.34s


6. Applications
Q: What are the practical applications of AI agents mentioned in the literature?
--------------------------------------------------
🔍 Searching for: 'What are the practical applications of AI agents mentioned in the literature?'


2025-09-23 12:45:53,981 - INFO - HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"


✓ Search completed in 2.60 seconds
📚 Found 3 relevant sources
A: AI agents have a wide range of practical applications across various industries. In healthcare, they assist with clinical diagnosis, decision support, mental health therapy, workflow optimization, and pharmaceutical research. In finance, they are used for forecasting and risk analysis. In scientific research, AI agents automate literature reviews and experimental design. They also play a role in s...
📚 Sources: 3 | ⏱️  Time: 2.60s



## 12. Interactive Query Interface

Let's create an interactive function that allows you to ask custom questions about the papers.


In [35]:
def ask_question(query_engine, question: str, show_sources: bool = True):
    """
    Ask a custom question to the RAG system and display results.
    
    Args:
        query_engine: The configured query engine
        question (str): Your question about the papers
        show_sources (bool): Whether to display source information
    """
    print(f"❓ Question: {question}")
    print("=" * 60)
    
    result = search_papers(query_engine, question, include_metadata=True)
    
    if result["success"]:
        print(f"💡 Answer:")
        print(result["response"])
        print(f"\n📊 Search completed in {result['search_time']:.2f} seconds")
        print(f"📚 Found {result['num_sources']} relevant sources")
        
        if show_sources and result["sources"]:
            print(f"\n📖 Source Details:")
            print("-" * 40)
            for i, source in enumerate(result["sources"], 1):
                print(f"\n{i}. Score: {source.get('score', 0):.3f}")
                print(f"   Text: {source['text'][:200]}...")
                
    else:
        print(f"❌ Error: {result['error']}")

# Example usage - you can modify this question
custom_question = "What are the key ethical considerations for AI agents?"

if query_engine:
    ask_question(query_engine, custom_question, show_sources=True)
else:
    print("❌ Query engine not available")


❓ Question: What are the key ethical considerations for AI agents?
🔍 Searching for: 'What are the key ethical considerations for AI agents?'


2025-09-23 12:46:02,247 - INFO - HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"


✓ Search completed in 2.64 seconds
📚 Found 3 relevant sources
💡 Answer:
Key ethical considerations for AI agents include ensuring accountability, fairness, and value alignment. In multi-agent systems, accountability gaps can arise when multiple agents interact to produce an outcome, making it challenging to assign responsibility for errors or unintended consequences. Additionally, bias propagation and amplification are significant concerns, as agents trained on biased data may reinforce each other's skewed decisions, leading to systemic inequities. Misalignment and value drift also pose risks, as agents may interpret objectives differently or optimize for local goals that diverge from human intent. Addressing these issues requires governance-aware architectures with role-based isolation, traceable decision logging, and participatory oversight mechanisms to maintain ethical integrity.

📊 Search completed in 2.64 seconds
📚 Found 3 relevant sources

📖 Source Details:
---------------------

## 13. System Performance and Statistics

Let's create functions to analyze and display performance statistics of our RAG system.


In [36]:
def display_system_stats(papers_list, vector_store, index):
    """
    Display comprehensive statistics about the RAG system.
    
    Args:
        papers_list: List of indexed papers
        vector_store: The vector store instance
        index: The vector index
    """
    print("📊 RAG System Statistics")
    print("=" * 50)
    
    # Paper statistics
    total_papers = len(papers_list)
    total_pages = sum(paper.get('page_count', 0) for paper in papers_list)
    total_size_mb = sum(paper.get('file_size_mb', 0) for paper in papers_list)
    
    print(f"📚 Document Statistics:")
    print(f"   Total Papers: {total_papers}")
    print(f"   Total Pages: {total_pages}")
    print(f"   Total Size: {total_size_mb:.2f} MB")
    print(f"   Average Pages per Paper: {total_pages/total_papers:.1f}" if total_papers > 0 else "   Average Pages: N/A")
    
    # Index statistics
    if index:
        print(f"\n🗂️  Index Statistics:")
        print(f"   Index Type: Vector Store Index")
        print(f"   Embedding Model: {get_config('api.openai.embedding_model', 'text-embedding-3-small')}")
        print(f"   LLM Model: {get_config('api.openai.model', 'gpt-4o-mini')}")
    
    # Storage paths
    print(f"\n💾 Storage Locations:")
    print(f"   Papers Folder: papers/agents")
    print(f"   Vector Database: storage/papers_vectordb")
    print(f"   Index Storage: storage/papers_index")
    
    # Recent papers by modification time
    if papers_list:
        print(f"\n📋 Paper Titles:")
        for i, paper in enumerate(papers_list, 1):
            title = paper['title']
            if len(title) > 50:
                title = title[:47] + "..."
            print(f"   {i}. {title}")

# Display system statistics
display_system_stats(papers_list, vector_store, index)
print("\n✅ RAG System Analysis Complete!")


📊 RAG System Statistics
📚 Document Statistics:
   Total Papers: 5
   Total Pages: 0
   Total Size: 37.92 MB
   Average Pages per Paper: 0.0

🗂️  Index Statistics:
   Index Type: Vector Store Index
   Embedding Model: text-embedding-3-small
   LLM Model: gpt-4o-mini

💾 Storage Locations:
   Papers Folder: papers/agents
   Vector Database: storage/papers_vectordb
   Index Storage: storage/papers_index

📋 Paper Titles:
   1. AI_Agents_vs_Agentic_AI.pdf
   2. Emerging_Agent_Architectures.pdf
   3. LLMReasoning_to_Autonomous_Agents.pdf
   4. Rise_and_Potential_LLM_Agents.pdf
   5. survey_of_self_evolving_agents.pdf

✅ RAG System Analysis Complete!


## Conclusion

🎉 **Congratulations!** You have successfully built a complete RAG (Retrieval-Augmented Generation) system for academic papers using LlamaIndex.

### What we accomplished:

1. **Environment Setup**: Configured API keys and dependencies
2. **Configuration Management**: Loaded system settings from YAML files
3. **LlamaIndex Configuration**: Set up embeddings, LLM, and text processing
4. **Vector Store**: Created a LanceDB vector database for storing embeddings
5. **Document Processing**: Loaded and processed PDF academic papers
6. **Vector Indexing**: Created searchable vector embeddings of documents
7. **Query Engine**: Set up retrieval and response generation
8. **Search Functions**: Implemented semantic search with metadata
9. **Paper Analysis**: Created functions for listing and summarizing papers
10. **Interactive Queries**: Built an interface for asking custom questions
11. **Performance Analytics**: Added system statistics and monitoring

### Key Features:

- **Semantic Search**: Find relevant content using natural language queries
- **Source Attribution**: Get detailed citations and references for answers
- **Metadata Integration**: Access paper titles, authors, and other metadata
- **Performance Monitoring**: Track search times and system statistics
- **Flexible Configuration**: Easy to modify models, chunk sizes, and parameters

### Next Steps:

1. **Experiment** with different queries to explore your document collection
2. **Modify** the `custom_question` variable to ask your own questions
3. **Adjust** parameters like `chunk_size`, `similarity_top_k` for different results
4. **Add** more papers to the `papers/agents` folder and rebuild the index
5. **Enhance** the system with additional features like filtering or ranking

### Usage Tips:

- Use specific, focused questions for better results
- Try different phrasings of the same question
- Check the source information to understand where answers come from
- Experiment with the `similarity_top_k` parameter to get more or fewer sources

Happy researching! 🔬📚
